In [1]:
import numpy as np 
import pandas as pd 
import os
from collections import Counter

# Loading Data

## Two Moons

In [2]:

def make_moons(n_per_class=500, noise=0.10, seed=42):
    rng = np.random.default_rng(seed)
    n = n_per_class

    t1 = np.linspace(0, np.pi, n)
    x1 = np.c_[np.cos(t1), np.sin(t1)]

    t2 = np.linspace(0, np.pi, n)
    x2 = np.c_[1 - np.cos(t2), 1 - np.sin(t2) - 0.5]

    X = np.vstack([x1, x2]) + rng.normal(scale=noise, size=(2*n, 2))
    y = np.hstack([np.zeros(n, dtype=int), np.ones(n, dtype=int)])

    return X, y


## Banana e Ripley

In [3]:
#!mkdir -p local_datasets

In [4]:
#!wget --quiet "https://raw.githubusercontent.com/sauloafoliveira/cclw-mlm/master/local_datasets/datasets-7627-10826-banana.csv" -O "local_datasets/datasets-7627-10826-banana.csv"

In [5]:
#!wget --quiet "https://raw.githubusercontent.com/sauloafoliveira/cclw-mlm/master/local_datasets/rip.csv" -O "local_datasets/rip.csv"

In [6]:
data_folder = "local_datasets"
banana = pd.read_csv(os.path.join(data_folder, "datasets-7627-10826-banana.csv"))
ripley = pd.read_csv(os.path.join(data_folder, "rip.csv"))

# Functions

In [7]:
def padronizar_zscore(X_treino, X_teste):
    """
    Padroniza os dados usando média e desvio padrão do treino.
    
    Parâmetros:
    - X_treino: dados de treino (n_amostras, n_features)
    - X_teste: dados de teste
    
    Retorna:
    - X_treino_padronizado, X_teste_padronizado
    """
    media = np.mean(X_treino, axis=0)
    desvio = np.std(X_treino, axis=0)
    
    desvio[desvio == 0] = 1
    
    X_treino_pad = (X_treino - media) / desvio
    X_teste_pad = (X_teste - media) / desvio
    
    return X_treino_pad, X_teste_pad

In [8]:
def calcular_distancia_euclidiana(ponto1, ponto2):
    return np.sqrt(np.sum((ponto1 - ponto2) ** 2))

def calcular_distancia_manhattan(ponto1, ponto2):
    return np.sum(np.abs(ponto1 - ponto2))

def calcular_matriz_distancias(X_teste, X_treino, metrica='euclidiana'):
    n_teste = X_teste.shape[0]
    n_treino = X_treino.shape[0]
    distancias = np.zeros((n_teste, n_treino))
    
    for i in range(n_teste):
        for j in range(n_treino):
            if metrica == 'euclidiana':
                distancias[i, j] = calcular_distancia_euclidiana(
                    X_teste[i], X_treino[j]
                )
            elif metrica == 'manhattan':
                distancias[i, j] = calcular_distancia_manhattan(
                    X_teste[i], X_treino[j]
                )
    
    return distancias

In [9]:
class KNN:
    
    def __init__(self, k=3, metrica='euclidiana', voto_ponderado=False):
        self.k = k
        self.metrica = metrica
        self.voto_ponderado = voto_ponderado
        self.X_treino = None
        self.y_treino = None
    
    def treinar(self, X_treino, y_treino):
        self.X_treino = X_treino
        self.y_treino = y_treino
    
    def encontrar_k_vizinhos(self, ponto_teste):
        distancias = np.zeros(len(self.X_treino))
        
        for i, ponto_treino in enumerate(self.X_treino):
            if self.metrica == 'euclidiana':
                distancias[i] = calcular_distancia_euclidiana(
                    ponto_teste, ponto_treino
                )
            elif self.metrica == 'manhattan':
                distancias[i] = calcular_distancia_manhattan(
                    ponto_teste, ponto_treino
                )
        
        indices_ordenados = np.argsort(distancias)
        k_indices = indices_ordenados[:self.k]
        k_distancias = distancias[k_indices]
        
        return k_indices, k_distancias
    
    def votar(self, k_rotulos, k_distancias):
        if not self.voto_ponderado:
            contagem = Counter(k_rotulos)
            return contagem.most_common(1)[0][0]
        else:
            classes_unicas = np.unique(k_rotulos)
            votos = {}
            
            for classe in classes_unicas:
                mascara = k_rotulos == classe
                distancias_classe = k_distancias[mascara]
                
                pesos = 1 / (distancias_classe + 1e-10)
                votos[classe] = np.sum(pesos)
            
            return max(votos, key=votos.get)
    
    def predizer(self, X_teste):
        """
        Prediz rótulos para conjunto de teste.
        """
        predicoes = np.zeros(len(X_teste))
        
        for i, ponto in enumerate(X_teste):
            k_indices, k_distancias = self.encontrar_k_vizinhos(ponto)
            k_rotulos = self.y_treino[k_indices]
            
            predicoes[i] = self.votar(k_rotulos, k_distancias)
        
        return predicoes.astype(int)

In [10]:
def calcular_metricas(y_verdadeiro, y_predito):
    y_verdadeiro = np.array(y_verdadeiro)
    y_predito = np.array(y_predito)
    
    classes = np.unique(y_verdadeiro)
    
    acuracia = np.mean(y_verdadeiro == y_predito)
    
    precisoes = []
    recalls = []
    f1s = []
    
    for classe in classes:
        tp = np.sum((y_verdadeiro == classe) & (y_predito == classe))
        
        fp = np.sum((y_verdadeiro != classe) & (y_predito == classe))
        
        fn = np.sum((y_verdadeiro == classe) & (y_predito != classe))
        
        precisao = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        f1 = 2 * (precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0
        
        precisoes.append(precisao)
        recalls.append(recall)
        f1s.append(f1)
    
    precisao_macro = np.mean(precisoes)
    recall_macro = np.mean(recalls)
    f1_macro = np.mean(f1s)
    
    return {
        'acuracia': acuracia,
        'precisao': precisao_macro,
        'recall': recall_macro,
        'f1': f1_macro
    }

In [11]:
class StratifiedKFold:
    """
    Implementação de K-Fold estratificado.
    Mantém proporção de classes em cada dobra.
    """
    
    def __init__(self, n_splits=5, random_state=42):
        self.n_splits = n_splits
        self.random_state = random_state
    
    def split(self, X, y):
        """
        Gera índices de treino/teste para cada dobra.
        
        Yields: (indices_treino, indices_teste)
        """
        np.random.seed(self.random_state)
        n_amostras = len(y)
        classes = np.unique(y)
        
        # Para cada classe, divide em dobras
        indices_por_classe = {}
        for classe in classes:
            indices_classe = np.where(y == classe)[0]
            np.random.shuffle(indices_classe)
            indices_por_classe[classe] = np.array_split(
                indices_classe, self.n_splits
            )
        
        # Gera dobras combinando classes
        for i in range(self.n_splits):
            indices_teste = []
            indices_treino = []
            
            for classe in classes:
                # Dobra i é teste, outras são treino
                indices_teste.extend(indices_por_classe[classe][i])
                
                for j in range(self.n_splits):
                    if j != i:
                        indices_treino.extend(indices_por_classe[classe][j])
            
            yield np.array(indices_treino), np.array(indices_teste)

# Results

In [12]:
def validacao_cruzada_knn(X, y, k_valores=[1, 3, 5, 7, 9], n_splits=5):
    """
    Realiza validação cruzada para diferentes valores de k.
    
    Retorna melhor k e métricas.
    """
    skf = StratifiedKFold(n_splits=n_splits, random_state=42)
    resultados = {}
    
    for k in k_valores:
        print(f"\nTestando k={k}...")
        metricas_dobras = []
        
        for fold, (idx_treino, idx_teste) in enumerate(skf.split(X, y)):
            # Separa dados
            X_treino, X_teste = X[idx_treino], X[idx_teste]
            y_treino, y_teste = y[idx_treino], y[idx_teste]
            
            # Padroniza
            X_treino_pad, X_teste_pad = padronizar_zscore(X_treino, X_teste)
            
            # Treina e prediz
            knn = KNN(k=k, metrica='euclidiana', voto_ponderado=False)
            knn.treinar(X_treino_pad, y_treino)
            predicoes = knn.predizer(X_teste_pad)
            
            # Calcula métricas
            metricas = calcular_metricas(y_teste, predicoes)
            metricas_dobras.append(metricas)
            
            print(f"  Fold {fold+1}: Acurácia={metricas['acuracia']:.4f}")
        
        # Agrega resultados
        resultados[k] = {
            'media': {
                'acuracia': np.mean([m['acuracia'] for m in metricas_dobras]),
                'precisao': np.mean([m['precisao'] for m in metricas_dobras]),
                'recall': np.mean([m['recall'] for m in metricas_dobras]),
                'f1': np.mean([m['f1'] for m in metricas_dobras])
            },
            'desvio': {
                'acuracia': np.std([m['acuracia'] for m in metricas_dobras]),
                'precisao': np.std([m['precisao'] for m in metricas_dobras]),
                'recall': np.std([m['recall'] for m in metricas_dobras]),
                'f1': np.std([m['f1'] for m in metricas_dobras])
            }
        }
    
    # Encontra melhor k baseado em F1
    melhor_k = max(resultados.keys(), 
                   key=lambda k: resultados[k]['media']['f1'])
    
    print(f"\n✓ Melhor k: {melhor_k}")
    print(f"F1-Score: {resultados[melhor_k]['media']['f1']:.4f} ± "
          f"{resultados[melhor_k]['desvio']['f1']:.4f}")
    
    return melhor_k, resultados

# Resultados

In [13]:
X_moons, y_moons = make_moons()
X_banana = banana.iloc[:, :-1].values
y_banana = banana.iloc[:, -1].values.astype(int)
X_ripley = ripley.iloc[:, :-1].values
y_ripley = ripley.iloc[:, -1].values.astype(int)

print(f"Two Moons: {X_moons.shape}")
print(f"Banana: {X_banana.shape}")
print(f"Ripley: {X_ripley.shape}")


import time

def teste_final(X, y, k, metrica='euclidiana', test_size=0.2):
    """Avaliação final em hold-out 20%"""
    np.random.seed(42)
    n = len(y)
    indices = np.arange(n)
    np.random.shuffle(indices)
    
    split_idx = int(n * (1 - test_size))
    idx_train = indices[:split_idx]
    idx_test = indices[split_idx:]
    
    X_train, X_test = X[idx_train], X[idx_test]
    y_train, y_test = y[idx_train], y[idx_test]
    
    X_train_pad, X_test_pad = padronizar_zscore(X_train, X_test)
    
    knn = KNN(k=k, metrica=metrica)
    knn.treinar(X_train_pad, y_train)
    
    tempos = []
    for _ in range(30):
        inicio = time.perf_counter()
        y_pred = knn.predizer(X_test_pad)
        fim = time.perf_counter()
        tempos.append((fim - inicio) * 1000 / len(X_test_pad))
    
    tempo_medio = np.mean(tempos)
    tempo_std = np.std(tempos)
    
    # Métricas
    metricas = calcular_metricas(y_test, y_pred)
    num_suportes = len(X_train)
    
    return {
        'metricas': metricas,
        'tempo_medio_ms': tempo_medio,
        'tempo_std_ms': tempo_std,
        'num_suportes': num_suportes
    }


def executar_competicao(nome, X, y):
    """Pipeline: validação -> teste -> resultados"""
    print("\n" + "="*60)
    print(f"  DATASET: {nome}")
    print("="*60)
    
    melhor_k, resultados_cv = validacao_cruzada_knn(X, y)
    
    print(f"\nTeste final com k={melhor_k}...")
    resultado_teste = teste_final(X, y, melhor_k)
    
    m = resultado_teste['metricas']
    print(f"\nRESULTADOS FINAIS:")
    print(f"  Acurácia:  {m['acuracia']:.4f}")
    print(f"  Precisão:  {m['precisao']:.4f}")
    print(f"  Recall:    {m['recall']:.4f}")
    print(f"  F1-Score:  {m['f1']:.4f}")
    print(f"  Tempo:     {resultado_teste['tempo_medio_ms']:.4f} ± {resultado_teste['tempo_std_ms']:.4f} ms/amostra")
    print(f"  Suportes:  {resultado_teste['num_suportes']}")
    
    return {
        'k': melhor_k,
        'validacao': resultados_cv,
        'teste': resultado_teste
    }


resultados = {}

print("INICIANDO EXPERIMENTOS\n")

resultados['two_moons'] = executar_competicao('Two Moons', X_moons, y_moons)
resultados['banana'] = executar_competicao('Banana', X_banana, y_banana)
resultados['ripley'] = executar_competicao('Ripley', X_ripley, y_ripley)


print("\n" + "█"*60)
print("█  RESUMO FINAL PARA RELATÓRIO")
print("█"*60)

for nome, res in resultados.items():
    m = res['teste']['metricas']
    print(f"\n{nome.upper()}:")
    print(f"  k = {res['k']}")
    print(f"  Acurácia:  {m['acuracia']:.4f}")
    print(f"  Precisão:  {m['precisao']:.4f}")
    print(f"  Recall:    {m['recall']:.4f}")
    print(f"  F1-Score:  {m['f1']:.4f}")
    print(f"  Tempo:     {res['teste']['tempo_medio_ms']:.4f} ± {res['teste']['tempo_std_ms']:.4f} ms")
    print(f"  Suportes:  {res['teste']['num_suportes']}")


import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
datasets = [
    ('Two Moons', X_moons, y_moons),
    ('Banana', X_banana, y_banana),
    ('Ripley', X_ripley, y_ripley)
]

for idx, (nome, X, y) in enumerate(datasets):
    ax = axes[idx]
    ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', alpha=0.6, label='Classe 0', s=20)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='red', alpha=0.6, label='Classe 1', s=20)
    ax.set_title(nome, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('datasets_visualization.png', dpi=150, bbox_inches='tight')
plt.show()


# CÉLULA 7: Salvar resultados em CSV
resultados_df = []
for nome, res in resultados.items():
    m = res['teste']['metricas']
    resultados_df.append({
        'dataset': nome,
        'k': res['k'],
        'acuracia': m['acuracia'],
        'precisao': m['precisao'],
        'recall': m['recall'],
        'f1': m['f1'],
        'tempo_ms': res['teste']['tempo_medio_ms'],
        'tempo_std_ms': res['teste']['tempo_std_ms'],
        'num_suportes': res['teste']['num_suportes']
    })

df = pd.DataFrame(resultados_df)
df.to_csv('resultados_knn.csv', index=False)
print("\n✓ Resultados salvos em: resultados_knn.csv")
print(df)

Two Moons: (1000, 2)
Banana: (5300, 2)
Ripley: (1248, 2)
INICIANDO EXPERIMENTOS


  DATASET: Two Moons

Testando k=1...
  Fold 1: Acurácia=1.0000
  Fold 2: Acurácia=1.0000
  Fold 3: Acurácia=1.0000
  Fold 4: Acurácia=1.0000
  Fold 5: Acurácia=1.0000

Testando k=3...
  Fold 1: Acurácia=1.0000
  Fold 2: Acurácia=1.0000
  Fold 3: Acurácia=1.0000
  Fold 4: Acurácia=1.0000
  Fold 5: Acurácia=1.0000

Testando k=5...
  Fold 1: Acurácia=1.0000
  Fold 2: Acurácia=1.0000
  Fold 3: Acurácia=1.0000
  Fold 4: Acurácia=1.0000
  Fold 5: Acurácia=1.0000

Testando k=7...
  Fold 1: Acurácia=1.0000
  Fold 2: Acurácia=1.0000
  Fold 3: Acurácia=1.0000
  Fold 4: Acurácia=1.0000
  Fold 5: Acurácia=1.0000

Testando k=9...
  Fold 1: Acurácia=1.0000
  Fold 2: Acurácia=1.0000
  Fold 3: Acurácia=1.0000
  Fold 4: Acurácia=1.0000
  Fold 5: Acurácia=1.0000

✓ Melhor k: 1
F1-Score: 1.0000 ± 0.0000

Teste final com k=1...

RESULTADOS FINAIS:
  Acurácia:  1.0000
  Precisão:  1.0000
  Recall:    1.0000
  F1-Score:  1.00

KeyboardInterrupt: 

In [ ]:
# CÉLULA DE VERIFICAÇÃO
print("Verificando Two Moons...")
print(f"Classes únicas: {np.unique(y_moons)}")
print(f"Distribuição: {np.bincount(y_moons)}")
print(f"Shape X: {X_moons.shape}")
print(f"Range X: [{X_moons.min():.2f}, {X_moons.max():.2f}]")

# Visualiza para ver se faz sentido
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], 
            c='blue', alpha=0.5, label='Classe 0')
plt.scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], 
            c='red', alpha=0.5, label='Classe 1')
plt.title('Two Moons - Visualização')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Teste manual com k=1
skf = StratifiedKFold(n_splits=5, random_state=42)
fold_1 = list(skf.split(X_moons, y_moons))[0]
idx_train, idx_test = fold_1

X_train = X_moons[idx_train]
X_test = X_moons[idx_test]
y_train = y_moons[idx_train]
y_test = y_moons[idx_test]

print(f"\nFold 1 - Tamanhos:")
print(f"Treino: {len(y_train)} | Teste: {len(y_test)}")
print(f"Distribuição treino: {np.bincount(y_train)}")
print(f"Distribuição teste: {np.bincount(y_test)}")

# Testa sem padronização
knn = KNN(k=1, metrica='euclidiana')
knn.treinar(X_train, y_train)
pred_sem_pad = knn.predizer(X_test)
acc_sem_pad = np.mean(y_test == pred_sem_pad)

# Testa com padronização
X_train_pad, X_test_pad = padronizar_zscore(X_train, X_test)
knn.treinar(X_train_pad, y_train)
pred_com_pad = knn.predizer(X_test_pad)
acc_com_pad = np.mean(y_test == pred_com_pad)

print(f"\nAcurácia SEM padronização: {acc_sem_pad:.4f}")
print(f"Acurácia COM padronização: {acc_com_pad:.4f}")